In [1]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Hardcoded SMILES for known HER2 inhibitors (positives)
positive_smiles = [
    'CS(=O)(=O)CCNCC1=CC=C(O1)C2=CC3=C(C=C2)N=CN=C3NC4=CC(=C(C=C4)OCC5=CC(=CC=C5)F)Cl',  # Lapatinib
    'CCOC1=C(C=C2C(=C1)N=CC(=C2NC3=CC(=C(C=C3)OCC4=CC=CC=N4)Cl)C#N)NC(=O)C=CCN(C)C',  # Neratinib
    'CC1=C(C=CC(=C1)NC2=NC=NC3=C2C=C(C=C3)NC4=NC(CO4)(C)C)OC5=CC6=NC=NN6C=C5',  # Tucatinib
    'CC1=CC(=CC=C1)NC(=O)NC2=CC(=C(C=C2)OC3=CC=C(C=C3)N4CCN(CC4)CC)Cl'  # Pyrotinib
]

# Hardcoded SMILES for non-inhibitors (negatives)
negative_smiles = [
    'CC(=O)OC1=CC=CC=C1C(=O)O',  # Aspirin
    'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',  # Ibuprofen
    'CC(=O)NC1=CC=C(C=C1)O',  # Paracetamol
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # Caffeine
    'CN(C)C(=N)NC(=N)N',  # Metformin
    'CC(C)C1=C(C(=C(N1CCC(CC(CC(=O)O)O)O)C2=CC=C(C=C2)F)C3=CC=CC=C3)C(=O)NC4=CC=CC=C4'  # Atorvastatin
]


In [2]:
# Generate Morgan fingerprints
def get_fingerprint(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    return np.array(fp)

# Collect features and labels
features = []
labels = []

for smi in positive_smiles:
    fp = get_fingerprint(smi)
    if fp is not None:
        features.append(fp)
        labels.append(1.0)

for smi in negative_smiles:
    fp = get_fingerprint(smi)
    if fp is not None:
        features.append(fp)
        labels.append(0.0)

X = np.array(features)
y = np.array(labels)

# Use sklearn's Random Forest classifier instead of PyTorch model
# You can also choose other models like LogisticRegression() or SVC(probability=True)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
print("Training sklearn model...")
model.fit(X, y)

# Evaluate the model (on training set)
train_predictions = model.predict(X)
train_accuracy = accuracy_score(y, train_predictions)
print(f'Training Accuracy: {train_accuracy:.4f}')


[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator
[14:06:34] DEPRECATION WARNING: please use MorganGenerator


Training sklearn model...
Training Accuracy: 1.0000


In [3]:
# Application in drug repurposing: Predict if Remdesivir (originally for COVID-19) interacts with HER2
remdesivir_smiles = 'CCC(CC)COC(=O)[C@H](C)N[P@](=O)(OC[C@H]1O[C@](C#N)([C@H](O)[C@@H]1O)C1=CNC2=C1C=NN2)OC1=CC=CC=C1'
rem_fp = get_fingerprint(remdesivir_smiles)

if rem_fp is not None:
    # Predict probability
    prediction_proba = model.predict_proba([rem_fp])[0][1]  # Get positive class probability
    prediction = model.predict([rem_fp])[0]  # Get predicted class

    print(f'\nDrug Repurposing Prediction for Remdesivir on HER2:')
    print(f'Predicted interaction probability: {prediction_proba:.4f}')
    if prediction_proba > 0.5:
        print('Potential interaction - Consider for repurposing in HER2-related diseases.')
    else:
        print('Low probability of interaction - Unlikely for repurposing in this context.')
else:
    print("Error: Could not generate fingerprint for Remdesivir")

# Display feature importance (only applicable to tree models)
if hasattr(model, 'feature_importances_'):
    print(f'\nTop 10 most important fingerprint bits:')
    feature_importance = model.feature_importances_
    top_indices = np.argsort(feature_importance)[-10:][::-1]
    for i, idx in enumerate(top_indices):
        print(f'{i+1}. Bit {idx}: {feature_importance[idx]:.4f}')

print("\nNote: This is a simplified example with a small dataset. In practice, use larger datasets and cross-validation.")



Drug Repurposing Prediction for Remdesivir on HER2:
Predicted interaction probability: 0.2300
Low probability of interaction - Unlikely for repurposing in this context.

Top 10 most important fingerprint bits:
1. Bit 875: 0.0452
2. Bit 329: 0.0338
3. Bit 561: 0.0338
4. Bit 322: 0.0337
5. Bit 1357: 0.0321
6. Bit 794: 0.0286
7. Bit 94: 0.0272
8. Bit 1097: 0.0235
9. Bit 191: 0.0217
10. Bit 491: 0.0206

Note: This is a simplified example with a small dataset. In practice, use larger datasets and cross-validation.


[14:06:40] DEPRECATION WARNING: please use MorganGenerator
